# FCC-ee HEB V26.2 ExtK20 impedance budget and tracking wake

Run FCC_ee_HEB_RF_IDDEFIX_Both_Planes_Full_Wake.ipynb first. Add its per-cavity RF impedance and 0.4 mm wakes to the existing V26.2 ExtK20 component tables. These tables supply the 150 nm NEG/Cu wall with 0.3 µm roughness, 20 extraction kicker modules, two damper striplines, wigglers, BPMs and bellows.

Run in the HEB package directory, repository root, or Example directory. The first code cell accepts a path to the HEB data directory and a separate path to the ExtK20 component tables. This notebook uses NumPy, SciPy, pandas and Matplotlib.

The new case is saved under data/total/v26_2_HOMonly_Damper_N2_RF_IDDEFIXfullwakeCST_ExtK20/. The ECSV ending in _ds_cell_exp235_positive_loss.ecsv is the Xwakes input for the workshop one-turn map.

## 1. Inputs and lattice normalization

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.constants import c, elementary_charge
from scipy.integrate import trapezoid

DATA_DIR = None       # e.g. Path('/path/to/fcc_ee_booster_pywit_model/data')
REFERENCE_DIR = None  # optional path to the existing ExtK20 export folder
RF_EXPORT_DIR = None  # optional path to the first notebook's RF export folder

BASE_TAG = 'v26_2_HOMonly_Damper_N2_RF_IDDEFIXdipoles_ExtK20'
CASE_TAG = 'v26_2_HOMonly_Damper_N2_RF_IDDEFIXfullwakeCST_ExtK20'
PREFIX = 'HEB_' + CASE_TAG
STEMS = {'long': 'long', 'x': 'dipx', 'y': 'dipy'}
FAMILIES = ('RW', 'Wiggler', 'Kicker', 'RF', 'BPM', 'Bellows', 'Damper')
NON_RF = tuple(name for name in FAMILIES if name != 'RF')
PARTS = ('Re', 'Im')
DISPLAY_NAMES = {'long': 'Longitudinal', 'x': 'Horizontal dipole', 'y': 'Vertical dipole'}
Z_UNITS = {'long': 'Ohm', 'x': 'Ohm/m', 'y': 'Ohm/m'}
W_UNITS = {'long': 'V/pC', 'x': 'V/(pC mm)', 'y': 'V/(pC mm)'}
SIGMA_WAKE_M = 0.4e-3
REFERENCE_ELEMENT = 'ds_cell_exp.235'
# Sliced Xsuite Twiss values used by the existing workshop wake loader.
MAP_BETA_M = {'x': 39.24504645, 'y': 149.75850350}

cwd = Path.cwd().resolve()
if DATA_DIR is None:
    candidates = (
        data for root in (cwd, *cwd.parents)
        for data in (root / 'data', root / 'fcc_ee_booster_pywit_model' / 'data')
    )
    DATA = next((data for data in candidates if (data / 'rf_cavities').is_dir()), None)
else:
    DATA = Path(DATA_DIR).expanduser().resolve()
if DATA is None or not (DATA / 'rf_cavities').is_dir():
    raise FileNotFoundError('Set DATA_DIR to the HEB package data directory containing rf_cavities/.')

BASE_DIR = (DATA / 'total' / BASE_TAG if REFERENCE_DIR is None
            else Path(REFERENCE_DIR).expanduser().resolve())
RF_DIR = (DATA / 'rf_cavities' / 'IDDEFIX_fullwake_CST' if RF_EXPORT_DIR is None
          else Path(RF_EXPORT_DIR).expanduser().resolve())
OPTICS = DATA / 'optics' / 'v26_2' / 'heb_ring_z.tfs'
OUT = DATA / 'total' / CASE_TAG
base_manifest_path = BASE_DIR / f'HEB_{BASE_TAG}_EXPORT_MANIFEST.json'
rf_manifest_path = RF_DIR / 'RF_IDDEFIX_fullwake_CST_manifest.json'
if not base_manifest_path.is_file():
    raise FileNotFoundError(f'ExtK20 manifest missing at {base_manifest_path}. Set REFERENCE_DIR to that export folder.')
if not rf_manifest_path.is_file():
    raise FileNotFoundError(f'RF manifest missing at {rf_manifest_path}. Run FCC_ee_HEB_RF_IDDEFIX_Both_Planes_Full_Wake.ipynb first.')
if not OPTICS.is_file():
    raise FileNotFoundError(f'V26.2 optics file missing at {OPTICS}.')

print(f'ExtK20 component tables  {BASE_DIR}')
print(f'Per-cavity RF input  {RF_DIR}')
print(f'Output directory  {OUT}')

In [ ]:
def file_hash(path):
    return sha256(path.read_bytes()).hexdigest()


def input_path(path):
    return path.relative_to(DATA).as_posix() if path.is_relative_to(DATA) else str(path)


base = json.loads(base_manifest_path.read_text(encoding='utf-8'))
rf = json.loads(rf_manifest_path.read_text(encoding='utf-8'))
optical_hash = file_hash(OPTICS)
if base['case_tag'] != BASE_TAG or base['optics_sha256'] != optical_hash:
    raise ValueError('The ExtK20 tables and V26.2 Twiss file must be from the same export.')
if (rf['schema_version'], rf['case'], rf['normalization'], rf['longitudinal_wake_sign']) != (
    1, 'IDDEFIX_fullwake_CST', 'per_cavity', 'CST'
):
    raise ValueError('Expected the per-cavity full-wake RF package with the CST longitudinal sign.')
if (base['wake_sigma_m'] != SIGMA_WAKE_M or rf['sigma_z_m'] != SIGMA_WAKE_M
    or rf['wake_units'] != W_UNITS or rf['impedance_units'] != Z_UNITS
    or rf['frequency_band_GHz'] != [0, 50]):
    raise ValueError('The RF and ring exports must use the expected units, band and 0.4 mm wake width.')

if rf['transverse_frequency_taper'] or rf['transverse_export_wake_length_m'] is not None:
    raise ValueError('Use the fully decayed transverse RF model fitted without a frequency taper.')
np.testing.assert_allclose(
    [base['rw_model']['NEG_thickness_m'], base['rw_model']['roughness_Rq_m']],
    [150e-9, 0.3e-6], rtol=1e-12, atol=0,
)

baseline_hashes = {input_path(base_manifest_path): file_hash(base_manifest_path)}
rf_manifest_hash = file_hash(rf_manifest_path)

def read_base_csv(suffix):
    path = BASE_DIR / f'HEB_{BASE_TAG}_{suffix}.csv'
    baseline_hashes[input_path(path)] = file_hash(path)
    return pd.read_csv(path, float_precision='round_trip')

inventory = read_base_csv('inventory')
provenance = read_base_csv('source_provenance')
damper_locations = read_base_csv('Damper_locations')
counts = inventory.set_index('component')['count']
np.testing.assert_array_equal(counts.loc[['RF', 'Kicker', 'Damper']], [112, 20, 2])
extractor_length_m = float(inventory.set_index('component').at['Kicker', 'physical_length_m'])
np.testing.assert_allclose(extractor_length_m, 20 * base['extraction_magnetic_length_per_module_m'])
lengths = inventory.set_index('component')['physical_length_m']
pipe_length_m = base['rw_circumference_m'] - lengths[['Kicker', 'Wiggler', 'Bellows']].sum()
np.testing.assert_allclose(lengths.at['RW'], pipe_length_m, atol=1e-6, rtol=0)
np.testing.assert_allclose(base['rw_pipe_length_m'], pipe_length_m, atol=1e-6, rtol=0)

multipliers = {plane: float(base['RF_ring_multipliers'][plane]) for plane in STEMS}
rf_rows = provenance['family'].eq('RF')
provided_rf_weights = provenance.loc[rf_rows].set_index('plane')['multiplier']
np.testing.assert_allclose([provided_rf_weights.at[p] for p in STEMS], list(multipliers.values()))
if multipliers['long'] != rf['n_cavities'] or multipliers['long'] != counts.at['RF']:
    raise ValueError('The RF fit and the ExtK20 inventory disagree on the cavity count.')

# The published dipole multiplier is sum(beta at RF cavities) / beta_budget.
with OPTICS.open(encoding='utf-8') as stream:
    columns = next(line.split()[1:] for line in stream if line.startswith('*'))
    key_index = {name: columns.index(name) for name in ('NAME', 'BETX', 'BETY')}
    cav_betas = []
    for line in stream:
        if 'ac800' in line.lower():
            fields = line.split()
            if 'ac800' in fields[key_index['NAME']].lower():
                cav_betas.append([float(fields[key_index['BETX']]), float(fields[key_index['BETY']])])
if len(cav_betas) != counts.at['RF']:
    raise ValueError(f'Expected {counts.at["RF"]} RF rows in the ExtK20 Twiss table. Found {len(cav_betas)}.')
reference_beta = {
    plane: float(beta_sum / multipliers[plane])
    for plane, beta_sum in zip(('x', 'y'), np.sum(cav_betas, axis=0))
}
map_scale = {plane: reference_beta[plane] / MAP_BETA_M[plane] for plane in reference_beta}
display(pd.DataFrame({
    'RF ring multiplier': multipliers,
    'Budget reference β [m]': reference_beta,
    'Workshop map β [m]': MAP_BETA_M,
    'Map wake multiplier': map_scale,
}).rename(index=DISPLAY_NAMES).round(8))
print(f"Rough NEG/Cu RW length  {pipe_length_m:,.3f} m")
print(f"Extraction magnetic length  {extractor_length_m:.2f} m across {int(counts.at['Kicker'])} modules")

## 2. Assemble the signed impedance and Gaussian wake

The new frequency grid includes the published component knots and the new RF adaptive knots. Interpolate the signed non-RF components onto this grid and add the new RF component. The wake uses the original ExtK20 distance grid and interpolates only the analytic RF wakes onto that grid. Both domains use the same per-plane RF scale.

In [ ]:
rf_hashes = {}

def load_rf_table(plane, quantity, ncols):
    record = rf['planes'][plane]['iddefix_fit'][quantity]
    path = RF_DIR / record['file']
    digest = file_hash(path)
    if digest != record['sha256']:
        raise ValueError(f'RF {quantity} hash mismatch for {path}. Rerun the RF notebook.')
    rf_hashes[input_path(path)] = digest
    values = np.loadtxt(path, ndmin=2)
    if (values.shape[1] != ncols or len(values) < 2 or not np.isfinite(values).all()
        or np.any(np.diff(values[:, 0]) <= 0)):
        raise ValueError(f'Invalid grid, column count or data in {path}')
    return path, values

rf_z = {}
rf_wake = {}
for plane in STEMS:
    z_path, z_values = load_rf_table(plane, 'impedance', 3)
    w_path, w_values = load_rf_table(plane, 'wake', 2)
    np.testing.assert_allclose(z_values[[0, -1], 0], [0, 50], rtol=0, atol=1e-9)
    rf_z[plane] = z_values
    rf_wake[plane] = w_values
    row = rf_rows & provenance['plane'].eq(plane)
    provenance.loc[row, ['path', 'wake_path']] = [input_path(z_path), input_path(w_path)]

old_z = {
    plane: read_base_csv('adaptive_long_budget' if plane == 'long' else f'component_budget_{stem}')
    for plane, stem in STEMS.items()
}
old_w = {plane: read_base_csv(f'component_wakes_{stem}_sigma0p4mm') for plane, stem in STEMS.items()}
s_mm = old_w['long']['s_mm'].to_numpy()
if len(s_mm) != base['wake_points'] or np.any(np.diff(s_mm) <= 0):
    raise ValueError('Unexpected distance grid in the ExtK20 wake tables.')
old_tracking_path = BASE_DIR / base['wake_file']
old_tracking_hash = file_hash(old_tracking_path)
if old_tracking_hash != base['wake_sha256']:
    raise ValueError('The ExtK20 ring wake does not match its original manifest.')
baseline_hashes[input_path(old_tracking_path)] = old_tracking_hash
old_ring = pd.read_csv(old_tracking_path, sep=r'\s+', comment='#').to_numpy()
np.testing.assert_allclose(
    old_ring, np.column_stack([s_mm * 1e6 / c, *(old_w[p]['Total'] for p in STEMS)]),
    atol=1e-10, rtol=2e-12,
)


def check_sum(table, prefix, atol):
    np.testing.assert_allclose(
        table[[prefix + component for component in FAMILIES]].sum(axis=1),
        table[prefix + 'Total'], atol=atol, rtol=2e-12,
    )


def build_budget(plane):
    original = old_z[plane]
    f_original = original['frequency_GHz'].to_numpy()
    f_rf, real_rf, imag_rf = rf_z[plane].T
    if not np.isfinite(original.to_numpy()).all() or np.any(np.diff(f_original) <= 0):
        raise ValueError(f'Invalid ExtK20 impedance grid or samples for {plane}.')
    np.testing.assert_allclose(f_original[[0, -1]], [0, 50], rtol=0, atol=1e-9)
    # Values closer than one mHz are duplicate frequency knots at this precision.
    f_new = np.union1d(f_original, f_rf)
    f_new = f_new[np.r_[True, np.diff(f_new) > 1e-12]]
    result = pd.DataFrame({'frequency_GHz': f_new})
    for part, z_values in zip(PARTS, (real_rf, imag_rf)):
        prefix = part + '_'
        check_sum(original, prefix, atol=1e-7)
        for family in FAMILIES:
            result[prefix + family] = (
                multipliers[plane] * np.interp(f_new, f_rf, z_values) if family == 'RF'
                else np.interp(f_new, f_original, original[prefix + family])
            )
        cols = [prefix + family for family in FAMILIES]
        result[prefix + 'Total'] = result[cols].sum(axis=1)
        expected_delta = result[prefix + 'RF'] - np.interp(f_new, f_original, original[prefix + 'RF'])
        np.testing.assert_allclose(
            result[prefix + 'Total'],
            np.interp(f_new, f_original, original[prefix + 'Total']) + expected_delta,
            atol=1e-7, rtol=2e-12,
        )
    return result


def build_wake(plane):
    original = old_w[plane]
    check_sum(original, '', atol=1e-9)
    np.testing.assert_allclose(original['s_mm'], s_mm, rtol=0, atol=1e-10)
    s_rf, w_rf = rf_wake[plane].T
    if not np.isfinite(original.to_numpy()).all() or s_rf[0] > s_mm[0] or s_rf[-1] < s_mm[-1]:
        raise ValueError(f'Wake data for {plane} must be finite and cover the ExtK20 distance grid.')
    result = original[['s_mm', *FAMILIES]].copy()
    result['RF'] = multipliers[plane] * np.interp(s_mm, s_rf, w_rf)
    result['Total'] = result[list(FAMILIES)].sum(axis=1)
    np.testing.assert_array_equal(result[list(NON_RF)], original[list(NON_RF)])
    np.testing.assert_allclose(
        result['Total'], original['Total'] + (result['RF'] - original['RF']),
        atol=1e-9, rtol=2e-12,
    )
    return result


impedance = {plane: build_budget(plane) for plane in STEMS}
wakes = {plane: build_wake(plane) for plane in STEMS}
wake_changes = pd.DataFrame([
    {
        'Plane': DISPLAY_NAMES[plane],
        'Units': W_UNITS[plane],
        'Previous RF peak |W|': np.max(np.abs(old_w[plane]['RF'])),
        'New RF peak |W|': np.max(np.abs(wakes[plane]['RF'])),
        'Maximum |ΔW| in total': np.max(np.abs(wakes[plane]['Total'] - old_w[plane]['Total'])),
    } for plane in STEMS
])
display(wake_changes.round(6))

## 3. Loss and kick factors

Use the real longitudinal impedance and the imaginary transverse impedance in the existing wake convention. For each component, integrate its signed term over 0–50 GHz with the squared Gaussian bunch spectrum. The dipole factors are reported per metre of drive offset.

In [ ]:
def gaussian_factor(frame, column, sigma_mm, unit_scale):
    f = frame['frequency_GHz'].to_numpy() * 1e9
    gaussian_squared = np.exp(-(2 * np.pi * f * sigma_mm * 1e-3 / c)**2)
    return float(2 * unit_scale * trapezoid(frame[column].to_numpy() * gaussian_squared, f))

factors = pd.DataFrame([
    {
        'plane': STEMS[plane],
        'component': family,
        'sigma_z_mm': sigma_mm,
        'units': W_UNITS[plane],
        'ExtK20': gaussian_factor(old_z[plane], part + family, sigma_mm, unit_scale),
        'Full-wake IDDEFIX RF': gaussian_factor(impedance[plane], part + family, sigma_mm, unit_scale),
    }
    for sigma_mm in (4.0, 2.62)
    for plane, part, unit_scale in (('long', 'Re_', 1e-12), ('x', 'Im_', 1e-15), ('y', 'Im_', 1e-15))
    for family in (*FAMILIES, 'Total')
])
display(factors.query("sigma_z_mm == 4.0").round(5))

long_rf_kappa = gaussian_factor(impedance['long'], 'Re_RF', 4.0, 1e-12)
np.testing.assert_allclose(
    long_rf_kappa, multipliers['long'] * rf['planes']['long']['loss_factor_4mm_V_per_pC_per_cavity'],
    rtol=5e-5, atol=1e-7,
)

for plane in ('x', 'y'):
    fitted = rf['planes'][plane]
    modeled = gaussian_factor(impedance[plane], 'Im_RF', 4.0, 1e-15)
    per_cavity = fitted['factor_checks_by_sigma_mm']['4']['frequency_integral']
    np.testing.assert_allclose(modeled, multipliers[plane] * per_cavity, rtol=5e-5, atol=1e-9)
    print(f"{DISPLAY_NAMES[plane]} RF  full-wake residual {fitted['wake_relative_l2']:.2%}  "
          f"bunch-spectrum residual {fitted['bunch_spectrum_relative_l2']:.2%}  "
          f"CST wake length {1e3 * fitted['wake_length_m']:.3f} mm")

## 4. Component budgets and wake comparison

In [ ]:
colors = {
    'RW': '#217188', 'Wiggler': '#805fb7', 'Kicker': '#2ca02c', 'RF': '#d62728',
    'BPM': '#e5a12e', 'Bellows': '#8c564b', 'Damper': '#1ca6a3',
}
labels = {
    'RW': 'Resistive wall', 'Wiggler': 'Wigglers', 'Kicker': 'Extraction kickers',
    'RF': 'RF cavities', 'BPM': 'BPMs', 'Bellows': 'Bellows', 'Damper': 'Damper striplines',
}
figures = {}

for plane, stem in STEMS.items():
    frame = impedance[plane]
    fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.7), layout='constrained')
    for ax, part in zip(axes, PARTS):
        for family in FAMILIES:
            y = frame[f'{part}_{family}'].to_numpy()
            ax.fill_between(frame['frequency_GHz'], 0., y, color=colors[family], alpha=.13, linewidth=0)
            ax.plot(frame['frequency_GHz'], y, color=colors[family], linewidth=1, label=labels[family])
        ax.plot(frame['frequency_GHz'], frame[f'{part}_Total'], color='#172c3b',
                lw=1.7, label='Total')
        y_unit = 'Ω' if plane == 'long' else 'Ω/m'
        ax.set(xlim=(0, 50), yscale='symlog', xlabel='Frequency [GHz]', ylabel=f'{part} Z [{y_unit}]')
        ax.grid(alpha=.22)
    axes[0].legend(ncol=2, frameon=False, fontsize=8)
    fig.suptitle(f'{DISPLAY_NAMES[plane]} impedance · rough NEG/Cu · V26.2 ExtK20')
    figures[f'impedance_{stem}'] = fig
    display(fig)
    plt.close(fig)

fig, axes = plt.subplots(3, 2, figsize=(12.8, 8.9), sharex=True, layout='constrained')
for row, plane in enumerate(STEMS):
    for ax, component in zip(axes[row], ('Total', 'RF')):
        ax.plot(s_mm, old_w[plane][component], '--', color='#536271', label='Published ExtK20')
        ax.plot(s_mm, wakes[plane][component], color='#bb4c21', label='Full-wake IDDEFIX RF')
        ax.set(title=f'{DISPLAY_NAMES[plane]} · {component}', ylabel=W_UNITS[plane])
        ax.grid(alpha=.22)
axes[0, 0].legend(frameon=False, loc='best')
for ax in axes[-1]:
    ax.set_xlabel('Trailing distance s [mm]')
fig.suptitle('V26.2 ExtK20 bunch wakes · 0.4 mm Gaussian · ring normalization')
figures['wake_comparison'] = fig
display(fig)
plt.close(fig)

## 5. Write the impedance and tracking files

The ring tables use the budget reference betas and the CST longitudinal sign. For the workshop input, multiply each dipole by \(\beta_{\mathrm{budget}}/\beta_{\mathrm{map}}\) and reverse the longitudinal sign so a positive value corresponds to energy loss in Xwakes. The time column uses ns. Longitudinal wakes use V/pC and dipoles use V/(pC·mm).

The Gaussian separation integral checks the mean energy kick of the exported 0.4 mm template for a 4 mm bunch at \(1.2\times10^{11}\) particles.

In [ ]:
columns = ('time', 'longitudinal', 'dipole_x', 'dipole_y')
ecsv_units = ('ns', 'V/pC', 'V / (pC mm)', 'V / (pC mm)')
ring_table = np.column_stack([s_mm * 1e6 / c, *(wakes[p]['Total'].to_numpy() for p in STEMS)])
workshop_table = ring_table.copy()
workshop_table[:, 1] *= -1.0
workshop_table[:, 2] *= map_scale['x']
workshop_table[:, 3] *= map_scale['y']

sigma_check_m = 4e-3
population_check = 1.2e11
separation_sigma_m = np.sqrt(2) * sigma_check_m
s_m = s_mm * 1e-3
density = np.exp(-0.5 * (s_m / separation_sigma_m)**2) / (np.sqrt(2 * np.pi) * separation_sigma_m)
check_loss_v_per_pc = float(trapezoid(workshop_table[:, 1] * density, s_m))
check_mean_kick_MeV = -1e6 * elementary_charge * population_check * check_loss_v_per_pc
if not np.isfinite(check_mean_kick_MeV) or check_mean_kick_MeV >= 0:
    raise ValueError(f'Workshop longitudinal wake must produce net energy loss. Mean kick {check_mean_kick_MeV:g} MeV.')
print(f'Gaussian mean wake kick at σz = 4 mm, Np = 1.2e11  {check_mean_kick_MeV:+.3f} MeV')

if file_hash(OPTICS) != optical_hash:
    raise RuntimeError('The optics file changed during the calculation. Rerun this notebook.')
if file_hash(rf_manifest_path) != rf_manifest_hash:
    raise RuntimeError('RF manifest changed during the calculation. Rerun this notebook.')
for name, digest in {**baseline_hashes, **rf_hashes}.items():
    if file_hash(DATA / name) != digest:
        raise RuntimeError(f'Input changed during the calculation: {name}')
OUT.mkdir(parents=True, exist_ok=True)

def write_ecsv(path, values, metadata):
    with path.open('w', encoding='utf-8') as stream:
        stream.write('# %ECSV 1.0\n# ---\n# datatype:\n')
        for name, unit in zip(columns, ecsv_units):
            stream.write(f'# - {{name: {name}, datatype: float64, unit: {json.dumps(unit)}}}\n')
        stream.write('# meta: !!omap\n')
        for key, value in metadata.items():
            stream.write(f'# - {{{key}: {json.dumps(value)}}}\n')
        stream.write('# schema: astropy-2.0\n' + ' '.join(columns) + '\n')
        np.savetxt(stream, values, fmt='%.16e')
    check = pd.read_csv(path, sep=r'\s+', comment='#', float_precision='round_trip')
    if tuple(check.columns) != columns or np.any(np.diff(check['time']) <= 0):
        raise ValueError(f'Invalid ECSV column names or time grid in {path}')
    np.testing.assert_allclose(check.to_numpy(), values, rtol=2e-15, atol=1e-15)


meta = {
    'wake_sigma_m': SIGMA_WAKE_M,
    'transverse_rf': 'IDDEFIX fit to complete CST bunch wake',
    'beta_reference_x_m': reference_beta['x'],
    'beta_reference_y_m': reference_beta['y'],
    'time_convention': 'Positive for trailing particles',
    'rf_manifest_sha256': rf_manifest_hash,
}
ring_path = OUT / f'{PREFIX}_sigma0p4mm_PyHEADTAIL.ecsv'
workshop_path = OUT / f'{PREFIX}_sigma0p4mm_ds_cell_exp235_positive_loss.ecsv'
write_ecsv(ring_path, ring_table, {**meta, 'longitudinal_convention': 'CST'})
write_ecsv(workshop_path, workshop_table, {
    **meta,
    'reference_element': REFERENCE_ELEMENT,
    'prepared_beta_x_m': MAP_BETA_M['x'],
    'prepared_beta_y_m': MAP_BETA_M['y'],
    'longitudinal_convention': 'Positive wake produces energy loss in Xwakes',
})
saved = [ring_path, workshop_path]

for plane, stem in STEMS.items():
    z = impedance[plane]
    w = wakes[plane]
    suffix = '_ADAPTIVE_0to50GHz' if plane == 'long' else ''
    z_path = OUT / f'Z_total_{stem}_{CASE_TAG}{suffix}.txt'
    w_path = OUT / f'W_total_{stem}_sigma0p4mm_{CASE_TAG}.txt'
    z_name = 'adaptive_long_budget' if plane == 'long' else f'component_budget_{stem}'
    z_csv = OUT / f'{PREFIX}_{z_name}.csv'
    w_csv = OUT / f'{PREFIX}_component_wakes_{stem}_sigma0p4mm.csv'
    np.savetxt(z_path, z[['frequency_GHz', 'Re_Total', 'Im_Total']].to_numpy(),
               fmt='%.14e', header=f'frequency_GHz Re_Z Im_Z  Signed ring total [{Z_UNITS[plane]}]')
    np.savetxt(w_path, w[['s_mm', 'Total']].to_numpy(),
               fmt='%.14e', header=f's_mm wake  Ring-normalized 0.4 mm Gaussian [{W_UNITS[plane]}]')
    z.to_csv(z_csv, index=False)
    w.to_csv(w_csv, index=False)
    saved.extend((z_path, w_path, z_csv, w_csv))

for name, frame in {
    'loss_and_kick_factors': factors,
    'inventory': inventory,
    'source_provenance': provenance,
    'Damper_locations': damper_locations,
}.items():
    path = OUT / f'{PREFIX}_{name}.csv'
    frame.to_csv(path, index=False)
    saved.append(path)
for name, fig in figures.items():
    path = OUT / f'{name}.pdf'
    fig.savefig(path)
    saved.append(path)

In [ ]:
# Carry over the original physical inventory and its assumptions.
physical_fields = {
    key: value for key, value in base.items()
    if key.startswith(('extraction_', 'rw_', 'stripline_', 'HEB_damper_', 'damper_weights',
                       'assumed_coll_', 'assumption_Coll_', 'active_m_per_kicker', 'warning_normalization'))
}
manifest = {
    **physical_fields,
    'case_tag': CASE_TAG,
    'date_utc': datetime.now(timezone.utc).isoformat(),
    'reference_case': BASE_TAG,
    'description': 'ExtK20 component sums with longitudinal HOM 1–3 and the complete-CST-wake transverse fit',
    'optics': input_path(OPTICS),
    'optics_sha256': optical_hash,
    'baseline_inputs_sha256': baseline_hashes,
    'RF_manifest': input_path(rf_manifest_path),
    'RF_manifest_sha256': rf_manifest_hash,
    'RF_source_inputs': rf['inputs'],
    'RF_matched_exports_sha256': rf_hashes,
    'RF_ring_multipliers': multipliers,
    'RF_loss_factor_4mm_V_per_pC_112cav': long_rf_kappa,
    'RF_fit_quality': {
        plane: {key: rf['planes'][plane][key] for key in (
            'bunch_spectrum_relative_l2', 'wake_relative_l2', 'wake_length_m', 'active_bound_hits', 'fitted_modes')}
        for plane in ('x', 'y')
    },
    'frequency_units': 'GHz',
    'impedance_units': Z_UNITS,
    'wake_units': W_UNITS,
    'wake_sigma_m': SIGMA_WAKE_M,
    'wake_points': len(s_mm),
    'wake_window_mm': [float(s_mm[0]), float(s_mm[-1])],
    'wake_df_Hz_for_nonRF_and_Damper': base['wake_df_Hz_for_nonRF_and_Damper'],
    'wake_fmax_Hz': base['wake_fmax_Hz'],
    'reference_element': REFERENCE_ELEMENT,
    'beta_budget_m': reference_beta,
    'beta_map_m': MAP_BETA_M,
    'map_dipole_multipliers': map_scale,
    'ring_wake_file': ring_path.name,
    'ring_wake_longitudinal_convention': 'CST',
    'wake_file': workshop_path.name,
    'wake_sha256': file_hash(workshop_path),
    'wake_longitudinal_convention': 'positive loss',
    'ecsv_time_units': 'ns',
    'mean_wake_kick_check': {
        'sigma_z_m': sigma_check_m, 'population': population_check, 'mean_energy_change_MeV': check_mean_kick_MeV,
    },
    'saved_files_sha256': {path.name: file_hash(path) for path in saved},
}
manifest_path = OUT / f'{PREFIX}_EXPORT_MANIFEST.json'
pending = manifest_path.with_suffix('.json.tmp')
pending.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
pending.replace(manifest_path)
print(f'Saved {len(saved) + 1} files to {OUT}')
print(f'Workshop ECSV  {workshop_path.name}')
print(f'SHA-256  {manifest["wake_sha256"]}')

The transverse RF terms in these totals come from a four-resonator fit to the complete finite CST bunch wake. The fully decayed impedance and the 0.4 mm tracking wake are model extrapolations from the 3 mm CST run.